# RepoExec BF16 serial generation on an FMLe A100 node

This is the second remote run. It uses four 14B-16B base code models in BF16 through vLLM. Inference is strictly serial: one model, one task, and one candidate request at a time.

The experiment keeps the first-run protocol: the first 30 `full_context` tasks, `raw`, `ast`, and `reduced_ast`, five sampled candidates, temperature 0.2, top-p 0.95, and 256 maximum output tokens. Docker evaluation remains on the local Windows machine.

## 1. Configuration

Run all cells in order. A disconnected browser does not stop a busy Jupyter kernel, but restarting or stopping the kernel does.

In [ ]:
import csv
import json
import os
import signal
import socket
import subprocess
import sys
import time
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen

if sys.version_info < (3, 10):
    raise RuntimeError(f"Python 3.10+ is required; this kernel uses {sys.version}")

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "repoexec_baseline").is_dir() and (path / "requirements-baseline.txt").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the cloned llm-experiment repository.")

STORAGE_ROOT = Path(os.environ.get("FMLE_STORAGE_ROOT", PROJECT_ROOT)).expanduser().resolve()
RUNS_ROOT = STORAGE_ROOT / "repoexec-runs-bf16"
VENV_DIR = STORAGE_ROOT / "repoexec-vllm-venv"
WORKER_PYTHON = VENV_DIR / "bin" / "python"
VLLM_BIN = VENV_DIR / "bin" / "vllm"
HF_HOME = STORAGE_ROOT / "hf-cache"
RUN_PREFIX = "fmle-bf16-serial30"
VLLM_HOST = "127.0.0.1"
VLLM_PORT = 8000
VLLM_BASE_URL = f"http://{VLLM_HOST}:{VLLM_PORT}/v1"

MODELS = [
    "deepseek-ai/DeepSeek-Coder-V2-Lite-Base",
    "bigcode/starcoder2-15b",
    "Qwen/Qwen2.5-Coder-14B",
    "bigcode/starcoder",
]
REPRESENTATIONS = ["raw", "ast", "reduced_ast"]
TASK_LIMIT = 30
NUM_RETURN_SEQUENCES = 5
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.2
TOP_P = 0.95
SEED = 42
MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.92
SERVER_START_TIMEOUT_SECONDS = 1800
REQUEST_TIMEOUT_SECONDS = 1800
DOWNLOAD_MODELS = True
VLLM_VERSION = "0.12.0"
PYTORCH_CUDA_INDEX = "https://download.pytorch.org/whl/cu128"

RUNTIME_CACHE_ROOT = STORAGE_ROOT / "runtime-cache"
TORCHINDUCTOR_CACHE_DIR = RUNTIME_CACHE_ROOT / "torchinductor"
TRITON_CACHE_DIR = RUNTIME_CACHE_ROOT / "triton"
for directory in (RUNS_ROOT, VENV_DIR.parent, HF_HOME, TORCHINDUCTOR_CACHE_DIR, TRITON_CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# FMLe containers can run with a numeric UID that has no /etc/passwd entry.
runtime_username = (
    os.environ.get("LOGNAME") or os.environ.get("USER")
    or os.environ.get("USERNAME") or f"fmle-uid-{os.getuid()}"
)
for variable in ("LOGNAME", "USER", "LNAME", "USERNAME"):
    os.environ[variable] = runtime_username
os.environ["XDG_CACHE_HOME"] = str(RUNTIME_CACHE_ROOT)
os.environ["TORCHINDUCTOR_CACHE_DIR"] = str(TORCHINDUCTOR_CACHE_DIR)
os.environ["TRITON_CACHE_DIR"] = str(TRITON_CACHE_DIR)
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_DATASETS_CACHE"] = str(HF_HOME / "datasets")
os.environ["HF_DATASETS_OFFLINE"] = "0"
os.environ["HF_HUB_OFFLINE"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
for variable in ("NO_PROXY", "no_proxy"):
    existing = {item.strip() for item in os.environ.get(variable, "").split(",") if item.strip()}
    os.environ[variable] = ",".join(sorted(existing | {"127.0.0.1", "localhost"}))

SESSION_STARTED_EPOCH = time.time()
TIMINGS = {}
MODEL_RUNTIME = {}
print(f"Host: {socket.gethostname()}")
print(f"Kernel Python: {sys.executable} ({sys.version.split()[0]})")
print(f"Project: {PROJECT_ROOT}")
print(f"Runtime user: {runtime_username} (uid={os.getuid()})")
print(f"Runtime cache: {RUNTIME_CACHE_ROOT}")
print(f"Runs: {RUNS_ROOT}")
print("Inference concurrency: exactly 1 request, 1 sequence, and 1 task")

## 2. Verify the dedicated GPU

This must show the assigned A100. Do not continue on a login node without a dedicated GPU.

In [ ]:
gpu_check = subprocess.run(["nvidia-smi", "-L"], text=True, capture_output=True)
if gpu_check.returncode != 0 or not gpu_check.stdout.strip():
    raise RuntimeError(f"No NVIDIA GPU is visible:\n{gpu_check.stderr}")
print(gpu_check.stdout)
subprocess.run(["nvidia-smi"], check=True)

## 3. Install the isolated vLLM environment

The Jupyter/RAPIDS environment is not modified. vLLM and RepoExec dependencies are installed in a dedicated virtual environment without `sudo`. The installed package versions are recorded in the final summary.

In [ ]:
started = time.perf_counter()
if not WORKER_PYTHON.is_file():
    subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)], check=True)
subprocess.run([str(WORKER_PYTHON), "-m", "pip", "install", "--upgrade", "pip", "wheel"], check=True)
subprocess.run(
    [str(WORKER_PYTHON), "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements-baseline.txt")],
    check=True,
)
subprocess.run(
    [
        str(WORKER_PYTHON), "-m", "pip", "install", "--upgrade",
        f"vllm=={VLLM_VERSION}", "huggingface_hub",
        "--extra-index-url", PYTORCH_CUDA_INDEX,
    ],
    check=True,
)
TIMINGS["python_dependencies_seconds"] = time.perf_counter() - started
version_script = (
    "import torch, vllm, transformers; "
    "print(torch.__version__, vllm.__version__, transformers.__version__, torch.version.cuda)"
)
versions = subprocess.check_output([str(WORKER_PYTHON), "-c", version_script], text=True).strip().split()
SOFTWARE_VERSIONS = {
    "torch": versions[0], "vllm": versions[1],
    "transformers": versions[2], "torch_cuda": versions[3],
}
if SOFTWARE_VERSIONS["vllm"] != VLLM_VERSION:
    raise RuntimeError(f"Expected vLLM {VLLM_VERSION}, got {SOFTWARE_VERSIONS['vllm']}")
if not SOFTWARE_VERSIONS["torch_cuda"].startswith("12.8"):
    raise RuntimeError(
        f"Expected a CUDA 12.8 PyTorch build, got {SOFTWARE_VERSIONS['torch_cuda']}"
    )
print(SOFTWARE_VERSIONS)
print(f"Dependency setup: {TIMINGS['python_dependencies_seconds'] / 60:.2f} min")

## 4. Authenticate and download BF16 checkpoints

`bigcode/starcoder` is gated. Accept its BigCode OpenRAIL-M agreement on Hugging Face before running this cell. Set `HF_TOKEN` in the instance environment, or enter it in the hidden prompt below. The token is never written to the notebook or result files.

All four checkpoints are downloaded sequentially into `HF_HOME`. They require substantial persistent storage.

In [ ]:
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("Hugging Face token: ").strip()
if not os.environ["HF_TOKEN"]:
    raise RuntimeError("HF_TOKEN is required for the gated bigcode/starcoder checkpoint.")

started = time.perf_counter()
if DOWNLOAD_MODELS:
    download_script = (
        "import sys; from huggingface_hub import snapshot_download; "
        "path = snapshot_download(repo_id=sys.argv[1]); print(path)"
    )
    for model in MODELS:
        print(f"Downloading or verifying {model}", flush=True)
        subprocess.run([str(WORKER_PYTHON), "-c", download_script, model], env=os.environ.copy(), check=True)
TIMINGS["model_download_seconds"] = time.perf_counter() - started
print(f"Model preparation: {TIMINGS['model_download_seconds'] / 60:.2f} min")

## 5. vLLM server and GPU telemetry helpers

Only one model is loaded at a time. `max_num_seqs=1` and the client sends one completion with `n=1`, so the five Pass@5 candidates are generated sequentially. The independent `nvidia-smi` process only records telemetry; it does not submit inference work.

In [ ]:
VLLM_PROCESS = None
VLLM_LOG_HANDLE = None
TELEMETRY_PROCESS = None
TELEMETRY_HANDLE = None

def slugify(value):
    return ''.join(character if character.isalnum() or character in '._-' else '-' for character in value)

def api_ready():
    try:
        with urlopen(f"{VLLM_BASE_URL}/models", timeout=2) as response:
            return response.status == 200
    except (OSError, URLError):
        return False

def start_telemetry(model):
    global TELEMETRY_PROCESS, TELEMETRY_HANDLE
    path = RUNS_ROOT / f"{RUN_PREFIX}-{slugify(model)}-gpu.csv"
    TELEMETRY_HANDLE = path.open('w', encoding='utf-8')
    TELEMETRY_HANDLE.write('timestamp,name,utilization_gpu_pct,power_draw_w,memory_used_mib\n')
    TELEMETRY_HANDLE.flush()
    TELEMETRY_PROCESS = subprocess.Popen(
        [
            'nvidia-smi',
            '--query-gpu=timestamp,name,utilization.gpu,power.draw,memory.used',
            '--format=csv,noheader,nounits',
            '-lms', '500',
        ],
        stdout=TELEMETRY_HANDLE, stderr=subprocess.STDOUT, start_new_session=True,
    )
    return path

def stop_telemetry():
    global TELEMETRY_PROCESS, TELEMETRY_HANDLE
    if TELEMETRY_PROCESS is not None and TELEMETRY_PROCESS.poll() is None:
        TELEMETRY_PROCESS.terminate()
        try:
            TELEMETRY_PROCESS.wait(timeout=10)
        except subprocess.TimeoutExpired:
            TELEMETRY_PROCESS.kill()
            TELEMETRY_PROCESS.wait(timeout=10)
    if TELEMETRY_HANDLE is not None and not TELEMETRY_HANDLE.closed:
        TELEMETRY_HANDLE.close()
    TELEMETRY_PROCESS = None
    TELEMETRY_HANDLE = None

def summarize_telemetry(path):
    if not path.exists():
        return {}
    with path.open(encoding='utf-8', errors='replace') as handle:
        rows = list(csv.DictReader(handle))
    def values(field):
        result = []
        for row in rows:
            try:
                result.append(float(row[field].strip()))
            except (KeyError, TypeError, ValueError):
                pass
        return result
    utilization = values('utilization_gpu_pct')
    power = values('power_draw_w')
    memory = values('memory_used_mib')
    return {
        'samples': len(rows),
        'mean_gpu_utilization_pct': sum(utilization) / len(utilization) if utilization else None,
        'max_gpu_utilization_pct': max(utilization) if utilization else None,
        'mean_power_draw_w': sum(power) / len(power) if power else None,
        'max_power_draw_w': max(power) if power else None,
        'max_memory_used_mib': max(memory) if memory else None,
    }

def stop_vllm_server():
    global VLLM_PROCESS, VLLM_LOG_HANDLE
    if VLLM_PROCESS is not None and VLLM_PROCESS.poll() is None:
        os.killpg(VLLM_PROCESS.pid, signal.SIGTERM)
        try:
            VLLM_PROCESS.wait(timeout=60)
        except subprocess.TimeoutExpired:
            os.killpg(VLLM_PROCESS.pid, signal.SIGKILL)
            VLLM_PROCESS.wait(timeout=30)
    if VLLM_LOG_HANDLE is not None and not VLLM_LOG_HANDLE.closed:
        VLLM_LOG_HANDLE.close()
    VLLM_PROCESS = None
    VLLM_LOG_HANDLE = None
    for _ in range(60):
        if not api_ready():
            break
        time.sleep(1)

def vllm_failure_excerpt(log_path, max_characters=60000):
    text = log_path.read_text(encoding='utf-8', errors='replace')
    markers = (
        'CUDA error', 'CUDA out of memory', 'OutOfMemoryError', 'RuntimeError:',
        'ValueError:', 'ImportError:', 'AssertionError:', 'Traceback (most recent call last)',
        'Killed', 'Segmentation fault',
    )
    positions = [text.find(marker) for marker in markers if text.find(marker) >= 0]
    start = max(0, min(positions) - 3000) if positions else max(0, len(text) - max_characters)
    return text[start:start + max_characters]

def start_vllm_server(model):
    global VLLM_PROCESS, VLLM_LOG_HANDLE
    stop_vllm_server()
    log_path = RUNS_ROOT / f"{RUN_PREFIX}-{slugify(model)}-vllm.log"
    VLLM_LOG_HANDLE = log_path.open('w', encoding='utf-8')
    command = [
        str(VLLM_BIN), 'serve', model,
        '--served-model-name', model,
        '--host', VLLM_HOST, '--port', str(VLLM_PORT),
        '--dtype', 'bfloat16',
        '--max-model-len', str(MAX_MODEL_LEN),
        '--max-num-seqs', '1',
        '--tensor-parallel-size', '1',
        '--gpu-memory-utilization', str(GPU_MEMORY_UTILIZATION),
        '--generation-config', 'vllm',
        '--enforce-eager',
        '--trust-remote-code',
    ]
    print('Starting:', ' '.join(command), flush=True)
    started = time.perf_counter()
    server_environment = os.environ.copy()
    if model.startswith('deepseek-ai/DeepSeek-Coder-V2'):
        server_environment['VLLM_ATTENTION_BACKEND'] = 'TRITON_MLA'
    else:
        server_environment.pop('VLLM_ATTENTION_BACKEND', None)
    VLLM_PROCESS = subprocess.Popen(
        command, cwd=PROJECT_ROOT, env=server_environment,
        stdout=VLLM_LOG_HANDLE, stderr=subprocess.STDOUT, start_new_session=True,
    )
    deadline = time.monotonic() + SERVER_START_TIMEOUT_SECONDS
    while time.monotonic() < deadline:
        if api_ready():
            elapsed = time.perf_counter() - started
            print(f"vLLM ready for {model} in {elapsed:.2f}s")
            return elapsed, log_path
        if VLLM_PROCESS.poll() is not None:
            break
        time.sleep(2)
    VLLM_LOG_HANDLE.flush()
    excerpt = vllm_failure_excerpt(log_path)
    stop_vllm_server()
    raise RuntimeError(
        f"vLLM did not start for {model}. Full log: {log_path}\n"
        f"Diagnostic excerpt:\n{excerpt}"
    )

def preflight(model):
    payload = json.dumps({
        'model': model, 'prompt': 'def add(a: int, b: int) -> int:\n    ',
        'max_tokens': 16, 'n': 1, 'temperature': 0.0, 'top_p': 1.0, 'seed': SEED,
    }).encode('utf-8')
    request = Request(f"{VLLM_BASE_URL}/completions", data=payload, headers={'Content-Type': 'application/json'})
    started = time.perf_counter()
    with urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as response:
        result = json.loads(response.read().decode('utf-8'))
    elapsed = time.perf_counter() - started
    print(f"Preflight {model}: {elapsed:.2f}s, usage={result.get('usage')}")
    return elapsed

## 6. Run the serial generation matrix

This performs 4 models x 3 representations x 30 tasks x 5 candidates = 1,800 sequential completion requests. Completed model/representation directories are skipped on rerun. An incomplete representation is regenerated from its beginning. Requests are never retried automatically.

Model loading happens once per model, not once per representation.

In [ ]:
import tarfile

def write_json(path, payload):
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    temporary.replace(path)

def load_json(path):
    if not path.exists():
        return None
    try:
        value = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        return None
    return value if isinstance(value, dict) else None

def count_rows(path):
    if not path.exists():
        return 0
    with path.open(encoding='utf-8') as handle:
        return sum(1 for line in handle if line.strip())

def run_slugify(value):
    return value.replace(':', '-').replace('/', '-')

def is_complete_run(output_dir, model, subset, representation, task_limit, candidate_count):
    config = load_json(output_dir / 'run_config.json')
    expected = {
        'model': model, 'backend': 'vllm', 'subset': subset,
        'representation': representation, 'task_count': task_limit,
        'num_return_sequences': candidate_count,
    }
    return bool(
        config and all(config.get(key) == value for key, value in expected.items())
        and count_rows(output_dir / 'processed_generations.jsonl') == task_limit
        and count_rows(output_dir / 'task_index.jsonl') == task_limit
        and count_rows(output_dir / 'task_metrics.jsonl') == task_limit * candidate_count
    )

def create_bundle(bundle_path, summary_path, runs_root, rows):
    filenames = (
        'generations.json', 'processed_generations.jsonl', 'task_metrics.jsonl',
        'task_index.jsonl', 'run_config.json', 'pre_eval_summary.json',
        'generation_timing.json',
    )
    temporary = bundle_path.with_suffix(bundle_path.suffix + '.tmp')
    with tarfile.open(temporary, 'w:gz') as archive:
        archive.add(summary_path, arcname=summary_path.name)
        for row in rows:
            run_dir = runs_root / row['run_dir']
            if row.get('status') not in ('completed', 'skipped_existing'):
                continue
            for filename in filenames:
                source = run_dir / filename
                if source.exists():
                    archive.add(source, arcname=f'{run_dir.name}/{filename}')
        for pattern in (f'{RUN_PREFIX}-*-gpu.csv', f'{RUN_PREFIX}-*-vllm.log'):
            for extra in runs_root.glob(pattern):
                archive.add(extra, arcname=extra.name)
    temporary.replace(bundle_path)

SUMMARY_PATH = RUNS_ROOT / f"{RUN_PREFIX}-generation-summary.json"
BUNDLE_PATH = RUNS_ROOT / f"{RUN_PREFIX}-generation-bundle.tar.gz"

def run_name(model, representation):
    return f"{RUN_PREFIX}-{run_slugify(model)}-full_context-{representation}-n{TASK_LIMIT}"

def completed_model_rows(model):
    rows = []
    for representation in REPRESENTATIONS:
        name = run_name(model, representation)
        output_dir = RUNS_ROOT / name
        if not is_complete_run(output_dir, model, 'full_context', representation, TASK_LIMIT, NUM_RETURN_SEQUENCES):
            return None
        timing = load_json(output_dir / 'generation_timing.json') or {}
        rows.append({
            'model': model, 'backend': 'vllm', 'subset': 'full_context',
            'representation': representation, 'task_limit': TASK_LIMIT,
            'num_return_sequences': NUM_RETURN_SEQUENCES,
            'ollama_parallel_requests': 1, 'parallel_tasks': 1,
            'run_dir': name, 'status': 'skipped_existing',
            'generation_wall_seconds': timing.get('generation_wall_seconds'),
        })
    return rows

def write_combined_summary(status, rows, error=None):
    now = time.time()
    summary = {
        'schema_version': 2, 'status': status, 'error': error,
        'started_at_utc': datetime.fromtimestamp(SESSION_STARTED_EPOCH, timezone.utc).isoformat(),
        'updated_at_utc': datetime.fromtimestamp(now, timezone.utc).isoformat(),
        'session_wall_seconds': now - SESSION_STARTED_EPOCH,
        'recorded_generation_wall_seconds': sum(float(row.get('generation_wall_seconds') or 0) for row in rows),
        'configuration': {
            'backend': 'vllm', 'dtype': 'bfloat16', 'models': MODELS,
            'representations': REPRESENTATIONS, 'subset': 'full_context',
            'task_limit': TASK_LIMIT, 'max_new_tokens': MAX_NEW_TOKENS,
            'num_return_sequences': NUM_RETURN_SEQUENCES, 'do_sample': True,
            'temperature': TEMPERATURE, 'top_p': TOP_P, 'seed': SEED,
            'max_model_len': MAX_MODEL_LEN, 'max_num_seqs': 1,
            'parallel_requests': 1, 'parallel_tasks': 1,
            'gpu_memory_utilization': GPU_MEMORY_UTILIZATION,
        },
        'software_versions': SOFTWARE_VERSIONS,
        'setup_timings': TIMINGS, 'model_runtime': MODEL_RUNTIME, 'runs': rows,
    }
    write_json(SUMMARY_PATH, summary)
    return summary

all_rows = []
matrix_started = time.perf_counter()
try:
    for model in MODELS:
        existing_rows = completed_model_rows(model)
        if existing_rows is not None:
            print(f"Skipping completed model: {model}")
            all_rows.extend(existing_rows)
            write_combined_summary('running', all_rows)
            continue

        telemetry_path = start_telemetry(model)
        server_started = time.perf_counter()
        try:
            startup_seconds, log_path = start_vllm_server(model)
            preflight_seconds = preflight(model)
            command = [
                str(WORKER_PYTHON), '-m', 'repoexec_baseline.run_generation_matrix',
                '--models', model, '--backend', 'vllm',
                '--representations', *REPRESENTATIONS, '--subset', 'full_context',
                '--task-limit', str(TASK_LIMIT), '--runs-root', str(RUNS_ROOT),
                '--run-prefix', RUN_PREFIX, '--max-new-tokens', str(MAX_NEW_TOKENS),
                '--num-return-sequences', str(NUM_RETURN_SEQUENCES), '--do-sample',
                '--temperature', str(TEMPERATURE), '--top-p', str(TOP_P), '--seed', str(SEED),
                '--parallel-requests', '1', '--parallel-tasks', '1',
                '--vllm-base-url', VLLM_BASE_URL,
                '--vllm-timeout-seconds', str(REQUEST_TIMEOUT_SECONDS), '--no-archive',
                '--session-start-epoch', str(SESSION_STARTED_EPOCH),
            ]
            print('Running serial matrix for', model, flush=True)
            subprocess.run(command, cwd=PROJECT_ROOT, env=os.environ.copy(), check=True)
            model_summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
            model_rows = model_summary['runs']
            all_rows.extend(model_rows)
            MODEL_RUNTIME[model] = {
                'server_startup_seconds': startup_seconds,
                'preflight_seconds': preflight_seconds,
                'loaded_model_wall_seconds': time.perf_counter() - server_started,
                'server_log': str(log_path), 'gpu_telemetry': str(telemetry_path),
            }
            write_combined_summary('running', all_rows)
        finally:
            stop_vllm_server()
            stop_telemetry()
            subprocess.run(['nvidia-smi'], check=False)
        MODEL_RUNTIME[model]['gpu_summary'] = summarize_telemetry(telemetry_path)
        write_combined_summary('running', all_rows)
finally:
    stop_vllm_server()
    stop_telemetry()

TIMINGS['generation_matrix_seconds'] = time.perf_counter() - matrix_started
summary = write_combined_summary('completed', all_rows)
create_bundle(BUNDLE_PATH, SUMMARY_PATH, RUNS_ROOT, all_rows)
print(f"Completed {len(all_rows)}/{len(MODELS) * len(REPRESENTATIONS)} runs")
print(f"Generation phase: {TIMINGS['generation_matrix_seconds'] / 3600:.2f} h")
print(f"Summary: {SUMMARY_PATH}")
print(f"Bundle: {BUNDLE_PATH}")

## 7. Inspect timings and download the bundle

In [ ]:
from IPython.display import FileLink, display

summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
total_active = time.time() - SESSION_STARTED_EPOCH
summary['notebook_active_seconds'] = total_active
summary['notebook_active_hours'] = total_active / 3600
write_json(SUMMARY_PATH, summary)
create_bundle(BUNDLE_PATH, SUMMARY_PATH, RUNS_ROOT, summary['runs'])
print(f"Status: {summary['status']}")
print(f"Completed runs: {len(summary['runs'])}/{len(MODELS) * len(REPRESENTATIONS)}")
print(f"Recorded generation: {summary['recorded_generation_wall_seconds'] / 3600:.2f} h")
print(f"Notebook active time: {total_active / 3600:.2f} h")
for model, runtime in summary.get('model_runtime', {}).items():
    print(model, runtime)
display(FileLink(str(BUNDLE_PATH)))

## 8. Local evaluation

After downloading and extracting the bundle, evaluate it on the local Windows machine:

```powershell
.\.venv\Scripts\python.exe -m repoexec_baseline.evaluate_generation_matrix `
  --matrix-summary remote-runs\fmle-bf16-serial30-generation-bundle\fmle-bf16-serial30-generation-summary.json `
  --repoexec-dir RepoExec
```

## 9. Optional cleanup

The long generation cell unloads every model automatically. This cell is safe to run after an interruption.

In [ ]:
stop_vllm_server()
stop_telemetry()
subprocess.run(['nvidia-smi'], check=False)
print('Notebook-owned vLLM and telemetry processes stopped.')